In [42]:
import pandas as pd
import numpy as np
import scipy.stats as spstats
import matplotlib.pyplot as plt
import seaborn as sns
import PIL.Image as Image
import os

from IPython.display import display

In [53]:
dataset_path_c1 = "../Datasets/Brain_Tumour_MRI_Detection/no"
dataset_path_c2 = "../Datasets/Brain_Tumour_MRI_Detection/yes"

In [111]:
# Utility function to create a list of file paths
def create_image_paths(dataset_path: str) -> list[str]:
    """
    Description
    -----------
    Creates a List of Str for the Image Files in the dataset path & Returns it (IF IT IS A FILE)
    """
    image_paths = []

    print(f"Collecting files from the path {dataset_path}")
    for file in os.scandir(dataset_path):
        image_paths.append(file.path)
    return image_paths

# Create the image paths list
c1_image_paths = create_image_paths(dataset_path_c1)
c2_image_paths = create_image_paths(dataset_path_c2)

# Display a small subset of image paths
display("Image Paths (First 5)")
display(c1_image_paths[:5])
display(c2_image_paths[:5])

# Display number of images
display(f"Number of Class 1 Images = {len(c1_image_paths)}")
display(f"Number of Class 2 Images = {len(c2_image_paths)}")

'Image Paths (First 5)'

['../Datasets/Brain_Tumour_MRI_Detection/no/0.jpg',
 '../Datasets/Brain_Tumour_MRI_Detection/no/1.jpg',
 '../Datasets/Brain_Tumour_MRI_Detection/no/2.jpg',
 '../Datasets/Brain_Tumour_MRI_Detection/no/3.jpg',
 '../Datasets/Brain_Tumour_MRI_Detection/no/4.jpg']

['../Datasets/Brain_Tumour_MRI_Detection/yes/0.jpg',
 '../Datasets/Brain_Tumour_MRI_Detection/yes/1.jpg',
 '../Datasets/Brain_Tumour_MRI_Detection/yes/2.jpg',
 '../Datasets/Brain_Tumour_MRI_Detection/yes/3.jpg',
 '../Datasets/Brain_Tumour_MRI_Detection/yes/4.jpg']

'Number of Class 1 Images = 98'

'Number of Class 2 Images = 155'

In [116]:
# ----- Train Test Splitting
# We'll use 80% of the samples for setting up the bayes classifier & 20% for validating it

n_train_samples_c1 = int(len(c1_image_paths) * 0.8)
n_train_samples_c2 = int(len(c2_image_paths) * 0.8)
# Train Samples
c1_train_image_paths = c1_image_paths[0:n_train_samples_c1]
c2_train_image_paths = c2_image_paths[0:n_train_samples_c2]
# Test Samples
c1_test_image_paths = c1_image_paths[n_train_samples_c1:]
c2_test_image_paths = c2_image_paths[n_train_samples_c2:]
# Total Samples
total_train_samples = len(c1_train_image_paths) + len(c2_train_image_paths)
total_test_samples = len(c1_test_image_paths) + len(c2_test_image_paths)
# Display Information
display(f"Number of Train Samples = {total_train_samples}")
display(f"Number of Test Samples = {total_test_samples}")
display(f"Number of Class 1 Train Samples = {len(c1_train_image_paths)}")
display(f"Number of Class 2 Train Samples = {len(c2_train_image_paths)}")
display(f"Number of Class 1 Test Samples = {len(c1_test_image_paths)}")
display(f"Number of Class 2 Test Samples = {len(c2_test_image_paths)}")

# ----- Calculation of Prior Probabilities
c1_prior = len(c1_train_image_paths) / total_train_samples
c2_prior = len(c2_train_image_paths) / total_train_samples
c1_prior_log = np.log(c1_prior)
c2_prior_log = np.log(c2_prior)

display(f"Prior of Class 1 = {c1_prior}")
display(f"Prior of Class 2 = {c2_prior}")
display(f"Log Prior of Class 1 = {c1_prior_log}")
display(f"Log Prior of Class 2 = {c2_prior_log}")
display(f"Total Prior = {c1_prior + c2_prior}")

'Number of Train Samples = 202'

'Number of Test Samples = 51'

'Number of Class 1 Train Samples = 78'

'Number of Class 2 Train Samples = 124'

'Number of Class 1 Test Samples = 20'

...

'Prior of Class 1 = 0.38613861386138615'

'Prior of Class 2 = 0.6138613861386139'

'Log Prior of Class 1 = -0.951558870711613'

'Log Prior of Class 2 = -0.4879861317961679'

'Total Prior = 1.0'

In [126]:
# Static Parameters

image_resolution = (64, 64)
eps = 1e-12

In [120]:
# Utility Functions

def convert_path_to_image_array(image_paths: list[str], resolution: tuple[int, int]):
    """
    Description
    -----------
    Utility function to open the image from paths & convert them into numpy arrays (Ravelled)
    """
    image_arrays = []
    for path in image_paths:
        image_object = Image.open(path).convert("L").resize(resolution)
        image_array = np.array(image_object, dtype=np.float64).ravel()
        image_arrays.append(image_array)
    return image_arrays

In [128]:
# I. Image itself will be our feature vector
"""
# Mathematical Framework
- Naive Per-Class Gaussian Independence Assumption
- Let's Assume Each Class in the image is a Gaussian Distribution which has a Sample Mean mu_j & sample 
variance sigma_j, where j is the index of the pixel. 
- Let's resize all the images to a dimension d, such that j = 0 -> j = D & All of them are indepedent 
of each other.
"""

c1_train_image_arrays = convert_path_to_image_array(c1_train_image_paths, image_resolution)
c2_train_image_arrays = convert_path_to_image_array(c2_train_image_paths, image_resolution)

# Calculate Mean
c1_cond_dist_mean_naive_1 = np.mean(c1_train_image_arrays)
c2_cond_dist_mean_naive_1 = np.mean(c1_train_image_arrays)
# Calculate Standard Deviation
c1_cond_dist_std_naive_1 = np.std(c1_train_image_arrays) + eps 
c2_cond_dist_std_naive_1 = np.std(c1_train_image_arrays) + eps

display(f"dim(x) = {c1_train_image_arrays[0].shape}")
display(f"P(x|C1) Naive 1= Normal({c1_cond_dist_mean_naive_1}, {c1_cond_dist_std_naive_1})")
display(f"P(x|C2) Naive 1= Normal({c2_cond_dist_mean_naive_1}, {c2_cond_dist_std_naive_1})")

# II. Image itself will be our feature vector
"""
# Mathematical Framework

- Naive Per-Pixel Gaussian Independence Assumption
- Let's Assume Each Pixel in the image is a Gaussian Distribution which has a Sample Mean mu_j & sample 
variance sigma_j, where j is the index of the pixel. 
- Let's resize all the images to a dimension d, such that j = 0 -> j = D & All of them are indepedent 
of each other.
"""

# Calculate Mean
c1_cond_dist_mean_naive_2 = np.mean(c1_train_image_arrays, axis=0)
c2_cond_dist_mean_naive_2 = np.mean(c2_train_image_arrays, axis=0)
# Calculate Standard Deviation
c1_cond_dist_std_naive_2 = np.std(c1_train_image_arrays, axis=0) + eps 
c2_cond_dist_std_naive_2 = np.std(c2_train_image_arrays, axis=0) + eps

display(f"P(x|C1) Naive 2 = Normal({c1_cond_dist_mean_naive_2}, {c1_cond_dist_std_naive_2})")
display(f"P(x|C2) Naive 2 = Normal({c2_cond_dist_mean_naive_2}, {c2_cond_dist_std_naive_2})")

'dim(x) = (4096,)'

'P(x|C1) Naive 1= Normal(53.622007712339745, 56.343034045144506)'

'P(x|C2) Naive 1= Normal(53.622007712339745, 56.343034045144506)'

'P(x|C1) Naive 2 = Normal([14.42307692 12.52564103 11.06410256 ...  9.05128205 10.29487179\n 13.71794872], [41.16711385 38.11108068 33.59232961 ... 30.31919822 34.97878807\n 43.08277585])'

'P(x|C2) Naive 2 = Normal([20.35483871 14.47580645 13.83870968 ... 12.91129032 15.75\n 24.12903226], [40.62717856 28.36668415 28.61405787 ... 27.68399214 32.02383815\n 53.32280994])'

In [149]:
# Prediction Utility Functions

def predict_image_likelihood(
    image_path: str, 
    resolution: tuple[int, int],
    c1_cond_dist_mean: np.ndarray,
    c2_cond_dist_mean: np.ndarray,
    c1_cond_dist_std: np.ndarray,
    c2_cond_dist_std: np.ndarray,
    projection: np.ndarray | None
):
    """
    Description
    -----------
    Predicts the image based on Log Likelihood 
    """
    image_object = Image.open(image_path).convert("L").resize(resolution)
    image_array = np.array(image_object, dtype=np.float64).ravel()
    if projection is not None:
        image_array  = image_array @ projection
    p_this_image_belongs_to_c1 = np.sum(spstats.norm.logpdf(image_array, c1_cond_dist_mean, c1_cond_dist_std))
    p_this_image_belongs_to_c2 = np.sum(spstats.norm.logpdf(image_array, c2_cond_dist_mean, c2_cond_dist_std))
    return "c1" if p_this_image_belongs_to_c1 > p_this_image_belongs_to_c2 else "c2"

def predict_image_bayesian(
    image_path: str, 
    resolution: tuple[int, int],
    c1_cond_dist_mean: np.ndarray,
    c2_cond_dist_mean: np.ndarray,
    c1_cond_dist_std: np.ndarray,
    c2_cond_dist_std: np.ndarray,
    c1_prior_log: np.ndarray,
    c2_prior_log: np.ndarray,
    projection: np.ndarray | None
):
    """
    Description
    -----------
    Predicts the image based on Posteriori
    """
    image_object = Image.open(image_path).convert("L").resize(resolution)
    image_array = np.array(image_object, dtype=np.float64).ravel()
    if projection is not None:
        image_array  = image_array @ projection
    p_this_image_belongs_to_c1 = np.sum(spstats.norm.logpdf(image_array, c1_cond_dist_mean, c1_cond_dist_std))
    p_this_image_belongs_to_c2 = np.sum(spstats.norm.logpdf(image_array, c2_cond_dist_mean, c2_cond_dist_std))
    p_this_image_belongs_to_c1 += c1_prior_log
    p_this_image_belongs_to_c2 += c2_prior_log
    return "c1" if p_this_image_belongs_to_c1 > p_this_image_belongs_to_c2 else "c2"

def predict_and_spit_accuracy(
    c1_cond_dist_mean: np.ndarray,
    c2_cond_dist_mean: np.ndarray,
    c1_cond_dist_std: np.ndarray,
    c2_cond_dist_std: np.ndarray,
    c1_prior_log: np.ndarray,
    c2_prior_log: np.ndarray,
    projection: np.ndarray | None
):
    """
    Description
    -----------
    Utility Function to Calculating Accuracy
    """
    correct_predictions_likelihood = 0
    correct_predictions_bayesian = 0
    # Perform Classification for C1 Test Images
    for image in c1_test_image_paths:
        output_likelihood = predict_image_likelihood(
            image,
            image_resolution,
            c1_cond_dist_mean,
            c2_cond_dist_mean,
            c1_cond_dist_std,
            c2_cond_dist_std,
            projection
        )
        output_bayesian = predict_image_bayesian(
            image,
            image_resolution,
            c1_cond_dist_mean,
            c2_cond_dist_mean,
            c1_cond_dist_std,
            c2_cond_dist_std,
            c1_prior_log,
            c2_prior_log,
            projection
        )

        if output_likelihood == "c1":
            correct_predictions_likelihood += 1
        if output_bayesian == "c1":
            correct_predictions_bayesian += 1
    # Perform Classification for C2 Test Images
    for image in c2_test_image_paths:
        output_likelihood = predict_image_likelihood(
            image,
            image_resolution,
            c1_cond_dist_mean,
            c2_cond_dist_mean,
            c1_cond_dist_std,
            c2_cond_dist_std,
            projection
        )
        output_bayesian = predict_image_bayesian(
            image,
            image_resolution,
            c1_cond_dist_mean,
            c2_cond_dist_mean,
            c1_cond_dist_std,
            c2_cond_dist_std,
            c1_prior_log,
            c2_prior_log,
            projection
        )

        if output_likelihood == "c2":
            correct_predictions_likelihood += 1
        if output_bayesian == "c2":
            correct_predictions_bayesian += 1

    accuracy_likelihood = correct_predictions_likelihood / total_test_samples
    accuracy_bayesian = correct_predictions_bayesian / total_test_samples
    print(f"Accuracy (Likelihood based Classification)= {accuracy_likelihood * 100}%")
    print(f"Accuracy (Bayesian based Classification)= {accuracy_bayesian * 100}%")

In [133]:
# Concatenate all the train samples along row
train_array = np.concatenate([np.array(c1_train_image_arrays), np.array(c2_train_image_arrays)], axis=0)
display("Train Array Shape : ", train_array.shape)

# Center around mean
train_array_mean_centered = train_array - np.mean(train_array, axis=0)

# Covariance matrix calculation
train_cov = train_array_mean_centered.T @ train_array_mean_centered / (train_array_mean_centered.shape[0] - 1)
display("Covariance Matrix Shape : ", train_cov.shape)

'Train Array Shape : '

(202, 4096)

'Covariance Matrix Shape : '

(4096, 4096)

In [134]:
# Eigenvalue & Eigenvector calculation for Covriance Matrix
eigenvalues, eigenvectors = np.linalg.eig(train_cov)

display("Number of Eigen Values = ", eigenvalues.shape)
display("Number of Eigen Vectors = ", eigenvectors.shape)

'Number of Eigen Values = '

(4096,)

'Number of Eigen Vectors = '

(4096, 4096)

In [ ]:
# Reverse Sort Eigen values 
sorted_eigenvalues = np.sort(eigenvalues)[::-1]
sorted_eigenvalues_indices = np.argsort(eigenvalues)[::-1]

# Normalization Constant
eigenvalues_sum = np.sum(sorted_eigenvalues)
# Error Thershold
projection_error_threshold = 0.95
# Prefix Sum Array
sorted_eigenvalues_prefix_sum = np.cumsum(sorted_eigenvalues)
# Selection Criteria in the Prefix Sum array
# We want all the elements where Normalized Prefix Sum is < Projection error Thershold 
criteria = sorted_eigenvalues_prefix_sum < projection_error_threshold * eigenvalues_sum
# Should be True ,...... False .... 
display("Criteria:", criteria)


principal_eigenvalues = sorted_eigenvalues[criteria]
n_principal_components = len(principal_eigenvalues)
principal_eigenvectors = eigenvectors.T[sorted_eigenvalues_indices[:n_principal_components]]

display("Principal Eigenvalues Shape", principal_eigenvalues.shape)
display("Principal Eigenvectors Shape", principal_eigenvectors.shape)

'Criteria:'

array([ True,  True,  True, ..., False, False, False], shape=(4096,))

'Principal Eigenvalues Shape'

(98,)

'Principal Eigenvectors Shape'

(98, 4096)

In [141]:
principal_projection_W = principal_eigenvectors

display("Shape of Projection Matrix", principal_projection_W.shape)

train_array_projected = train_array_mean_centered @ principal_projection_W.T

display(train_array_mean_centered.shape)
display(train_array_projected.shape)

'Shape of Projection Matrix'

(98, 4096)

(202, 4096)

(202, 98)

In [142]:
c1_train_image_arrays_projected = train_array_projected[0:n_train_samples_c1]
c2_train_image_arrays_projected = train_array_projected[n_train_samples_c1:]

display(c1_train_image_arrays_projected.shape)
display(c2_train_image_arrays_projected.shape)

(78, 98)

(124, 98)

In [143]:
# III. Image itself will be our feature vector (Using Principle Component Analysis)

# Calculate Mean
c1_cond_dist_mean_projected = np.mean(c1_train_image_arrays_projected, axis=0)
c2_cond_dist_mean_projected = np.mean(c2_train_image_arrays_projected, axis=0)
# Calculate Standard Deviation
c1_cond_dist_std_projected = np.std(c1_train_image_arrays_projected, axis=0) + eps 
c2_cond_dist_std_projected = np.std(c2_train_image_arrays_projected, axis=0) + eps

display(f"dim(x) = {c1_train_image_arrays_projected[0].shape}")
display(f"P(x|C1) = Normal({c1_cond_dist_mean_projected}, {c1_cond_dist_std_projected})")
display(f"P(x|C2) = Normal({c2_cond_dist_mean_projected}, {c2_cond_dist_std_projected})")

'dim(x) = (98,)'

'P(x|C1) = Normal([-8.07270296e+02+0.j -1.49454379e+01+0.j  2.46186561e+02+0.j\n -4.98687684e+01+0.j -5.68305200e+01+0.j -2.75181845e+01+0.j\n  5.95371269e+01+0.j -2.03473371e+01+0.j -1.33567993e+01+0.j\n  3.63546102e+01+0.j  3.03718812e+01+0.j -3.16889179e+00+0.j\n  8.82173049e+01+0.j -9.92335888e+00+0.j -3.14948701e+01+0.j\n -3.48743398e+01+0.j  3.10849981e+01+0.j -3.90042975e+00+0.j\n -2.63535296e+01+0.j  4.62941192e+01+0.j -1.34322846e+01+0.j\n -2.51346757e+01+0.j -4.21218897e+01+0.j  2.22329148e+01+0.j\n -7.81012102e+00+0.j -4.52521329e+00+0.j  3.61055400e+01+0.j\n -2.57602469e+01+0.j -3.42510165e+01+0.j  2.80965842e+01+0.j\n -2.50860878e+01+0.j -2.90361946e+00+0.j  1.43071331e+00+0.j\n  2.30535984e+01+0.j -2.14089290e+01+0.j -1.36556494e+01+0.j\n  5.20089057e+00+0.j  1.25384174e+00+0.j  2.46096596e+00+0.j\n -1.66359083e+01+0.j -1.13832266e+01+0.j -3.42675632e+00+0.j\n -2.32128073e+01+0.j -5.14864323e+00+0.j  2.10542453e+01+0.j\n -3.68485847e+01+0.j  1.99942621e+01+0.j -4.61725706

'P(x|C2) = Normal([ 5.07799057e+02+0.j  9.40116252e+00+0.j -1.54859289e+02+0.j\n  3.13690640e+01+0.j  3.57482303e+01+0.j  1.73098257e+01+0.j\n -3.74507734e+01+0.j  1.27991314e+01+0.j  8.40185764e+00+0.j\n -2.28682226e+01+0.j -1.91048930e+01+0.j  1.99333516e+00+0.j\n -5.54915305e+01+0.j  6.24211285e+00+0.j  1.98112892e+01+0.j\n  2.19370847e+01+0.j -1.95534666e+01+0.j  2.45349613e+00+0.j\n  1.65772202e+01+0.j -2.91204943e+01+0.j  8.44934034e+00+0.j\n  1.58105218e+01+0.j  2.64960274e+01+0.j -1.39852206e+01+0.j\n  4.91281806e+00+0.j  2.84650514e+00+0.j -2.27115494e+01+0.j\n  1.62040263e+01+0.j  2.15449943e+01+0.j -1.76736578e+01+0.j\n  1.57799585e+01+0.j  1.82647031e+00+0.j -8.99964821e-01+0.j\n -1.45014570e+01+0.j  1.34669070e+01+0.j  8.58984396e+00+0.j\n -3.27152794e+00+0.j -7.88706903e-01+0.j -1.54802698e+00+0.j\n  1.04645229e+01+0.j  7.16041676e+00+0.j  2.15554026e+00+0.j\n  1.46016046e+01+0.j  3.23866267e+00+0.j -1.32437995e+01+0.j\n  2.31789484e+01+0.j -1.25770358e+01+0.j  2.90440364

In [152]:
display("Naive Assumption 1 (Per Class Gaussian)")
predict_and_spit_accuracy(
    c1_cond_dist_mean_naive_1,
    c2_cond_dist_mean_naive_1,
    c1_cond_dist_std_naive_1,
    c2_cond_dist_std_naive_1,
    c1_prior_log,
    c2_prior_log,
    None
)
display("Naive Assumption 2 (Per Pixel Gaussian)")
predict_and_spit_accuracy(
    c1_cond_dist_mean_naive_2,
    c2_cond_dist_mean_naive_2,
    c1_cond_dist_std_naive_2,
    c2_cond_dist_std_naive_2,
    c1_prior_log,
    c2_prior_log,
    None
)
display("Projected Along Principal Component")
predict_and_spit_accuracy(
    c1_cond_dist_mean_projected,
    c2_cond_dist_mean_projected,
    c1_cond_dist_std_projected,
    c2_cond_dist_std_projected,
    c1_prior_log,
    c2_prior_log,
    principal_projection_W.T
)

'Naive Assumption 1 (Per Class Gaussian)'

Accuracy (Likelihood based Classification)= 60.78431372549019%
Accuracy (Bayesian based Classification)= 60.78431372549019%


'Naive Assumption 2 (Per Pixel Gaussian)'

Accuracy (Likelihood based Classification)= 72.54901960784314%
Accuracy (Bayesian based Classification)= 72.54901960784314%


'Projected Along Principal Component'

Accuracy (Likelihood based Classification)= 70.58823529411765%
Accuracy (Bayesian based Classification)= 70.58823529411765%
